In [1]:
import kagglehub
from seaborn import heatmap

# Download latest version
path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\jains\.cache\kagglehub\datasets\yelp-dataset\yelp-dataset\versions\4


In [2]:
import pandas as pd


In [3]:
import seaborn as sns
import  matplotlib.pyplot as plt


In [4]:
reviews = pd.concat(
    pd.read_json(f'{path}/yelp_academic_dataset_review.json',
                 lines=True, chunksize=100_000, nrows=500_000)
)

In [5]:
business = pd.concat(
    pd.read_json(f'{path}/yelp_academic_dataset_business.json',
                 lines=True, chunksize=100_000, nrows=500_000)
)

In [6]:
user = pd.concat(
    pd.read_json(f'{path}/yelp_academic_dataset_user.json',
                 lines=True, chunksize=100_000, nrows=500_000)
)

In [7]:
tip = pd.concat(
    pd.read_json(f'{path}/yelp_academic_dataset_tip.json',
                 lines=True, chunksize=100_000, nrows=500_000)
)

In [8]:
checkin = pd.read_json(f'{path}/yelp_academic_dataset_checkin.json',
                       lines=True, convert_dates=False)

In [ ]:
(reviews == '').sum() + (reviews == 'None').sum()


In [ ]:
(checkin == '').sum() + (checkin == 'None').sum()


In [ ]:

(user == '').sum() + (user == 'None').sum()

In [ ]:

(tip == '').sum() + (tip == 'None').sum()

In [ ]:

(business == '').sum() + (business == 'None').sum()

In [ ]:
import  numpy as np
business["missingAddress"] = business["address"].isin(['', 'None']).astype(int)
business["missingpostal_code"]= business["postal_code"].isin(['', 'None']).astype(int)

In [ ]:
business["categories"].value_counts().reset_index().sort_values("count",ascending=False)

In [ ]:
business

In [9]:
reviews=reviews.drop(columns=["cool","funny"])

In [ ]:
reviews

In [10]:
user=user.drop(columns=["funny","cool","elite","friends","fans","compliment_cool","compliment_funny","compliment_plain","compliment_photos","compliment_profile","compliment_cute"])

In [ ]:
user


In [11]:
business = business.dropna(subset=["categories"]).copy()

In [12]:
business["cat_list"] = business["categories"].str.split(", ")
exploded = business.explode("cat_list")
exploded["cat_list"] = exploded["cat_list"].str.strip()

In [13]:
YELP_TO_FSQ = {
    # ── spot types ──
    "Bakeries":                    (13002, "Bakery"),
    "Bars":                        (13003, "Bar"),
    "Breakfast & Brunch":          (13028, "Breakfast Spot"),
    "Cafes":                       (13032, "Café"),
    "Coffee & Tea":                (13034, "Coffee Shop"),
    "Bubble Tea":                  (13033, "Bubble Tea Shop"),
    "Desserts":                    (13040, "Dessert Shop"),
    "Ice Cream & Frozen Yogurt":   (13046, "Ice Cream Parlor"),
    "Donuts":                      (13043, "Donut Shop"),
    "Juice Bars & Smoothies":      (13059, "Juice Bar"),
    "Fast Food":                   (13145, "Fast Food Restaurant"),
    "Food Court":                  (13052, "Food Court"),
    "Food Trucks":                 (13054, "Food Truck"),
    "Pizza":                       (13064, "Pizzeria"),
    "Steakhouses":                 (13383, "Steakhouse"),

    # ── cuisines ──
    "American (New)":              (13068, "American"),
    "American (Traditional)":      (13068, "American"),
    "Asian Fusion":                (13072, "Asian"),
    "Chinese":                     (13099, "Chinese"),
    "Indian":                      (13199, "Indian"),
    "Italian":                     (13236, "Italian"),
    "Japanese":                    (13263, "Japanese"),
    "Korean":                      (13289, "Korean"),
    "Mediterranean":               (13302, "Mediterranean"),
    "Mexican":                     (13303, "Mexican"),
    "Middle Eastern":              (13306, "Middle Eastern"),
    "Thai":                        (13352, "Thai"),
    "Vietnamese":                  (13377, "Vietnamese"),
    "Seafood":                     (13338, "Seafood"),
    "Sushi Bars":                  (13276, "Sushi"),
    "Vegan":                       (13385, "Vegan / Vegetarian"),
    "Vegetarian":                  (13385, "Vegan / Vegetarian"),
}

print(f"{len(YELP_TO_FSQ)} Yelp category names mapped")


32 Yelp category names mapped


In [14]:
exploded["fsqid"]    = exploded["cat_list"].map(lambda c: YELP_TO_FSQ.get(c, (None, None))[0])
exploded["fsq_name"] = exploded["cat_list"].map(lambda c: YELP_TO_FSQ.get(c, (None, None))[1])

In [15]:
mapped = exploded.dropna(subset=["fsqid"])

In [ ]:
mapped

In [16]:
reviews["wordsInDesc"]=reviews["text"].str.split().str.len()

In [ ]:
reviews.loc[reviews["wordsInDesc"].between(1,4),"text"]

In [17]:
from ftlangdetect import detect as ft_detect

def fast_lang(text):
    if not isinstance(text, str) or len(text.strip()) < 20:
        return None
    return ft_detect(text.replace("\n", " "))["lang"]

reviews["lang"] = reviews["text"].map(fast_lang)   # minutes, not hours
reviews = reviews[reviews["lang"] == "en"]

In [ ]:
reviews.loc[reviews["wordsInDesc"].between(3,24),"text"]

In [18]:
reviews_3UP = reviews[reviews["wordsInDesc"]>=4]

In [ ]:
reviews_3UP

In [ ]:
checkin


In [19]:
checkin["checkinCount"] = checkin["date"].str.split(",").str.len()
checkin

,business_id,date,checkinCount
0,---kPU91CF4Lq2-WlRu9Lw,"2020-03-13 21:10:56, 2020-06-02 22:18:06, 2020...",11
1,--0iUa4sNDFiZFrAdIWhZQ,"2010-09-13 21:43:09, 2011-05-04 23:08:15, 2011...",10
2,--30_8IhuyMHbSOcNWd6DQ,"2013-06-14 23:29:17, 2014-08-13 23:20:22",2
3,--7PUidqRWpRSpXebiyxTg,"2011-02-15 17:12:00, 2011-07-28 02:46:10, 2012...",10
4,--7jw19RH9JKXgFohspgQw,"2014-04-21 20:42:11, 2014-04-28 21:04:46, 2014...",26
...,...,...,...
131925,zznJox6-nmXlGYNWgTDwQQ,"2013-03-23 16:22:47, 2013-04-07 02:03:12, 2013...",67
131926,zznZqH9CiAznbkV6fXyHWA,2021-06-12 01:16:12,1
131927,zzu6_r3DxBJuXcjnOYVdTw,"2011-05-24 01:35:13, 2012-01-01 23:44:33, 2012...",23
131928,zzw66H6hVjXQEt0Js3Mo4A,"2016-12-03 23:33:26, 2018-12-02 19:08:45",2


In [20]:
split_dates = checkin["date"].str.split(", ")

checkin["firstCheckin"]  = pd.to_datetime(split_dates.str[0])
checkin["latestCheckin"] = pd.to_datetime(split_dates.str[-1])
checkin

,business_id,date,checkinCount,firstCheckin,latestCheckin
0,---kPU91CF4Lq2-WlRu9Lw,"2020-03-13 21:10:56, 2020-06-02 22:18:06, 2020...",11,2020-03-13 21:10:56,2021-11-11 16:23:50
1,--0iUa4sNDFiZFrAdIWhZQ,"2010-09-13 21:43:09, 2011-05-04 23:08:15, 2011...",10,2010-09-13 21:43:09,2014-04-12 23:04:47
2,--30_8IhuyMHbSOcNWd6DQ,"2013-06-14 23:29:17, 2014-08-13 23:20:22",2,2013-06-14 23:29:17,2014-08-13 23:20:22
3,--7PUidqRWpRSpXebiyxTg,"2011-02-15 17:12:00, 2011-07-28 02:46:10, 2012...",10,2011-02-15 17:12:00,2015-09-27 13:18:32
4,--7jw19RH9JKXgFohspgQw,"2014-04-21 20:42:11, 2014-04-28 21:04:46, 2014...",26,2014-04-21 20:42:11,2021-06-21 19:59:50
...,...,...,...,...,...
131925,zznJox6-nmXlGYNWgTDwQQ,"2013-03-23 16:22:47, 2013-04-07 02:03:12, 2013...",67,2013-03-23 16:22:47,2021-05-16 21:43:26
131926,zznZqH9CiAznbkV6fXyHWA,2021-06-12 01:16:12,1,2021-06-12 01:16:12,2021-06-12 01:16:12
131927,zzu6_r3DxBJuXcjnOYVdTw,"2011-05-24 01:35:13, 2012-01-01 23:44:33, 2012...",23,2011-05-24 01:35:13,2013-12-13 00:58:14
131928,zzw66H6hVjXQEt0Js3Mo4A,"2016-12-03 23:33:26, 2018-12-02 19:08:45",2,2016-12-03 23:33:26,2018-12-02 19:08:45


In [21]:
checkin=checkin.drop(columns=["date"])

In [ ]:
checkin

In [22]:
food_biz = (mapped[["business_id", "fsqid", "fsq_name", "stars"]]
            .drop_duplicates("business_id")
            .rename(columns={"stars": "biz_stars"}))
reviews_3UP = reviews_3UP.merge(food_biz, on="business_id", how="inner")

In [23]:
restaurant_docs = (reviews_3UP.groupby("business_id").agg(all_text=("text", lambda x: " ".join(x.head(50))),avg_stars=("stars", "mean"),n_reviews=("text", "count")).reset_index())

In [24]:
import os
os.makedirs("data", exist_ok=True)

reviews_3UP.to_parquet("data/reviews_3UP.parquet")
restaurant_docs.to_parquet("data/restaurant_docs.parquet")
reviews.to_parquet("data/reviews_clean.parquet")
business.to_parquet("data/business_clean.parquet")
mapped.to_parquet("data/mapped.parquet")
user.to_parquet("data/user_clean.parquet")
tip.to_parquet("data/tip_clean.parquet")
checkin.to_parquet("data/checkin_clean.parquet")

print("Saved:", os.listdir("data"))

Saved: ['business_clean.parquet', 'checkin_clean.parquet', 'mapped.parquet', 'restaurant_docs.parquet', 'reviews_3UP.parquet', 'reviews_clean.parquet', 'tip_clean.parquet', 'user_clean.parquet']


In [25]:
import pandas as pd
df = pd.read_parquet("data/reviews_3UP.parquet")
print(df.shape)
df.head()

(350439, 12)


,review_id,user_id,business_id,stars,useful,text,date,wordsInDesc,lang,fsqid,fsq_name,biz_stars
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,101,en,13028.0,Breakfast Spot,3.0
1,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30,55,en,13028.0,Breakfast Spot,3.5
2,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,40,en,13199.0,Indian,4.0
3,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15,94,en,13003.0,Bar,4.0
4,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31,65,en,13302.0,Mediterranean,4.0
